# Edible — Baseline Model Training

Fine-tunes EfficientNet-B0 on the Texas wild-berry dataset.

**Safety principle**: toxic-class misclassification as edible is the critical failure mode.  
The training loop monitors *toxic FP rate* and saves a separate `best_safety.pt` checkpoint.

## Quick-start (Colab)
1. Set runtime → GPU (T4 is fine)
2. Mount Google Drive (cell below)
3. Set `REPO_PATH` and `DATA_PATH` to match your Drive layout
4. Run all cells

## Local training (TODO: migrate to Colab)
```
make install-dev  # installs ml extras too
uv run python -c "from edible.model.train import TrainConfig, train; train(TrainConfig())"
```

In [ ]:
# ── 1. Detect environment ────────────────────────────────────────────────────
import os
import sys

IN_COLAB = 'google.colab' in sys.modules
print(f'Running in Colab: {IN_COLAB}')

In [ ]:
# ── 2. Mount Google Drive (Colab only) ──────────────────────────────────────
# TODO: update REPO_PATH and DATA_PATH to match your Drive layout
REPO_PATH = '/content/drive/MyDrive/edible'   # path to cloned repo
DATA_PATH = '/content/drive/MyDrive/edible/data'  # path to data/images and data/species.json

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd {REPO_PATH}
else:
    # Local: repo root is two levels up from this notebook
    REPO_PATH = os.path.dirname(os.path.dirname(os.path.abspath('train_baseline.ipynb')))
    DATA_PATH = os.path.join(REPO_PATH, 'data')
    os.chdir(REPO_PATH)

print(f'Repo:  {REPO_PATH}')
print(f'Data:  {DATA_PATH}')

In [ ]:
# ── 3. Install dependencies ──────────────────────────────────────────────────
if IN_COLAB:
    !pip install -q uv
    !uv sync --extra dev --extra ml
    # Add src/ to Python path so `import edible` works
    sys.path.insert(0, os.path.join(REPO_PATH, 'src'))

import torch
print(f'PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')

In [ ]:
# ── 4. Configure training ────────────────────────────────────────────────────
from pathlib import Path
from edible.model.classifier import ClassifierConfig
from edible.model.train import TrainConfig

cfg = TrainConfig(
    images_dir=Path(DATA_PATH) / 'images',
    species_db_path=Path(DATA_PATH) / 'species.json',
    classifier_config=ClassifierConfig(
        model_name='efficientnet_b0',
        pretrained=True,
        dropout=0.3,
        toxic_loss_multiplier=2.0,
    ),
    epochs=30,
    batch_size=32,
    learning_rate=1e-3,
    checkpoint_dir=Path(REPO_PATH) / 'checkpoints',
    toxic_fp_patience=5,
)

print(cfg)

In [ ]:
# ── 5. Dataset stats ─────────────────────────────────────────────────────────
from edible.model.dataset import EdibleDataset
from torchvision import transforms

t = transforms.Compose([transforms.Resize((32, 32)), transforms.ToTensor()])

for split in ('train', 'val', 'test'):
    ds = EdibleDataset(cfg.images_dir, cfg.species_db_path, split=split, transform=t)
    print(f'{split:5s}: {len(ds):5d} images across {ds.num_classes()} classes')

ds_all = EdibleDataset(cfg.images_dir, cfg.species_db_path, split=None, transform=t)
print(f'\nClass map:')
for i, sid in enumerate(ds_all.species_ids()):
    toxic = '(TOXIC)' if i in ds_all.toxic_class_indices() else ''
    print(f'  {i:2d}: {sid} {toxic}')

In [ ]:
# ── 6. Train ─────────────────────────────────────────────────────────────────
from edible.model.train import train

history = train(cfg)

In [ ]:
# ── 7. Plot training curves ───────────────────────────────────────────────────
import matplotlib.pyplot as plt

epochs = [r.epoch for r in history]
val_acc = [r.val_accuracy for r in history]
toxic_fp = [r.toxic_fp_rate for r in history]
train_loss = [r.train_loss for r in history]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs, train_loss, 'b-o', markersize=3)
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch')

axes[1].plot(epochs, val_acc, 'g-o', markersize=3)
axes[1].set_title('Val Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylim(0, 1)

axes[2].plot(epochs, toxic_fp, 'r-o', markersize=3, label='Toxic FP rate')
axes[2].axhline(0.05, color='orange', linestyle='--', label='Alarm threshold (5%)')
axes[2].set_title('Toxic False-Positive Rate (SAFETY)')
axes[2].set_xlabel('Epoch')
axes[2].set_ylim(0, 1)
axes[2].legend()

plt.tight_layout()
plt.savefig(Path(REPO_PATH) / 'checkpoints' / 'training_curves.png', dpi=120)
plt.show()

In [ ]:
# ── 8. Evaluate best checkpoint on test set ───────────────────────────────────
from edible.model.classifier import build_classifier
from edible.model.evaluate import evaluate_model
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

test_ds = EdibleDataset(cfg.images_dir, cfg.species_db_path, split='test')
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

# Load best-safety checkpoint
ckpt = torch.load(cfg.checkpoint_dir / 'best_safety.pt', map_location=device)
cfg.classifier_config.num_classes = test_ds.num_classes()
model = build_classifier(cfg.classifier_config).to(device)
model.load_state_dict(ckpt['model_state_dict'])

metrics = evaluate_model(
    model, test_loader, device,
    test_ds.num_classes(), test_ds.toxic_class_indices(), test_ds.species_ids()
)

print('=== Test Set Safety Report ===')
print(metrics.safety_summary())
print()
print(metrics.classification_report())